# 06 — Computer Vision (Defect Classification)

This notebook covers the **Computer Vision** item from the project roadmap (item 05): classifying the type of defect visible in a photograph of a component — a scratch, a crack, contamination, and so on. This is the project's required **multimodal core**.

**Scope correction (2026-08-19):** the roadmap originally described this item as "read the equipment nameplate / fault code from a photo". That does not match the official bootcamp spec, which requires defect-*type* classification over the **MVTec Anomaly Detection** dataset, with a classification-accuracy metric on a held-out test set. See `README.md`'s Compliance checklist and `CLAUDE.md`'s 2026-08-19 note for the full reasoning. Nameplate/fault-code photo reading is not built here.

**Dataset — curated, not generic:** MVTec AD has 15 generic object/texture categories (bottle, hazelnut, toothbrush, ...), most irrelevant to a motor/VFD copilot. This notebook only uses 4 categories physically plausible on or around motors and VFDs: `cable`, `metal_nut`, `screw`, `transistor`.

This becomes the **Vision Tool** in the target architecture — a future Diagnostic Agent will consult it alongside the RAG Tool and the History Tool. Combining them is a later roadmap step (the Orchestrator Agent); this notebook only builds and evaluates the standalone tool.

Unlike notebook 05, this notebook makes real OpenAI API calls (the zero-shot baseline) and needs `OPENAI_API_KEY` to run end to end.

In [1]:
import sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from factory_floor.config import DEFECT_IMAGE_DIR, DEFECT_MANIFEST_CSV
from factory_floor.defect_dataset import (
    MVTEC_CATEGORIES,
    COARSE_LABELS,
    COARSE_LABEL_MAP,
    scan_raw_images,
    build_manifest,
    save_manifest,
    load_manifest,
    manifest_summary,
)
from factory_floor.vision import (
    extract_features_batch,
    train_classifier,
    save_classifier,
    classify_defect_trained,
    classify_defect_zero_shot,
    evaluate_classifier,
)


/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Dataset — the curated MVTec AD subset

`download_defect_images.py` fetches the 4 curated categories from the `Voxel51/mvtec-ad` mirror on the Hugging Face Hub (the official mvtec.com download page gates the combined archive behind a license form that doesn't respond to a plain scripted request — see the script's docstring / `CLAUDE.md` for the full story) and organizes them as `data/defect_images/raw/<category>/<mvtec_split>/<native_defect_label>/`.

Run `python download_defect_images.py` from the project root first if the cell below reports the images are missing.

In [2]:
raw_dir = DEFECT_IMAGE_DIR / 'raw'
missing = [c for c in MVTEC_CATEGORIES if not (raw_dir / c / 'test').exists()]
if missing:
    print(f'Missing categories: {missing}')
    print('Run: python download_defect_images.py  (from the project root)')
else:
    print(f'All 4 categories present under {raw_dir}')


All 4 categories present under /Users/marcelocorreia/Desktop/Factory_Floor_Chatbot/data/defect_images/raw


## 2. Manifest and stratified split

MVTec's *native* train/test split only has `good` images in `train/` (it was designed for unsupervised anomaly detection, not supervised classification) — unusable on its own to train a defect-type classifier. `scan_raw_images()` pools every labelled image from all native splits together, resolves each one to a generic coarse label via `COARSE_LABEL_MAP` (confirmed against the real downloaded subfolder names, not guessed from memory — see `factory_floor/defect_dataset.py`), and `build_manifest()` then makes our **own** stratified 75/25 train/test split (fixed seed) over those coarse labels.

In [3]:
rows = scan_raw_images()
print(f'{len(rows)} labelled images found across {MVTEC_CATEGORIES}')

from collections import Counter
native_counts = Counter((r['category'], r['native_defect_label']) for r in rows)
for key in sorted(native_counts):
    coarse = COARSE_LABEL_MAP[key]
    print(f'  {key[0]:>12} / {key[1]:<24} {native_counts[key]:>4}  -> coarse: {coarse}')


1502 labelled images found across ['cable', 'metal_nut', 'screw', 'transistor']
         cable / bent_wire                  13  -> coarse: deformation
         cable / cable_swap                 12  -> coarse: other_defect
         cable / combined                   11  -> coarse: other_defect
         cable / cut_inner_insulation       14  -> coarse: structural_damage
         cable / cut_outer_insulation       10  -> coarse: structural_damage
         cable / good                      282  -> coarse: good
         cable / missing_cable              12  -> coarse: other_defect
         cable / missing_wire               10  -> coarse: other_defect
         cable / poke_insulation            10  -> coarse: structural_damage
     metal_nut / bent                       25  -> coarse: deformation
     metal_nut / color                      22  -> coarse: contamination
     metal_nut / flip                       23  -> coarse: deformation
     metal_nut / good                      242  -> 

In [4]:
manifest_rows = build_manifest(rows, test_size=0.25, seed=42)
save_manifest(manifest_rows, DEFECT_MANIFEST_CSV)

summary = manifest_summary(manifest_rows)
print(f'Manifest saved to {DEFECT_MANIFEST_CSV} ({len(manifest_rows)} rows)\n')
print(f'{"coarse_label":<20}{"train":>8}{"test":>8}')
for label in COARSE_LABELS:
    print(f'{label:<20}{summary[(label, "train")]:>8}{summary[(label, "test")]:>8}')


Manifest saved to /Users/marcelocorreia/Desktop/Factory_Floor_Chatbot/defect_image_manifest.csv (1502 rows)

coarse_label           train    test
good                     868     290
scratch                   54      18
deformation               53      18
structural_damage         75      25
contamination             17       5
other_defect              59      20


## 3. Baseline — zero-shot vision LLM

No training needed: `classify_defect_zero_shot()` sends the photo straight to the same `gpt-4.1-mini` model already used for text answers in `factory_floor/rag.py`, with a prompt restricted to the 6 coarse labels and structured output. This is the "inexpensive, general-purpose" baseline the trained classifier (section 4) is benchmarked against.

Evaluating on the *full* test split (~375 images) is affordable within the project's €25 budget but slow and skewed toward the dominant `good` class during everyday development. `ZERO_SHOT_SAMPLE_SIZE` below caps the sample per class (stratified) for fast iteration — set it to `None` to run the full held-out test set for the official evaluation deliverable.

In [5]:
import random

ZERO_SHOT_SAMPLE_SIZE = 15  # per coarse_label; set to None for the full test split

test_rows = [r for r in manifest_rows if r['split'] == 'test']
if ZERO_SHOT_SAMPLE_SIZE is not None:
    rng = random.Random(42)
    by_label = {}
    for r in test_rows:
        by_label.setdefault(r['coarse_label'], []).append(r)
    zero_shot_rows = []
    for label, label_rows in by_label.items():
        rng.shuffle(label_rows)
        zero_shot_rows.extend(label_rows[:ZERO_SHOT_SAMPLE_SIZE])
else:
    zero_shot_rows = test_rows

print(f'Evaluating zero-shot baseline on {len(zero_shot_rows)} of {len(test_rows)} test images')


Evaluating zero-shot baseline on 80 of 376 test images


In [6]:
zero_shot_eval = evaluate_classifier(classify_defect_zero_shot, zero_shot_rows)
print(f"Zero-shot LLM accuracy: {zero_shot_eval['accuracy']:.1%} "
      f"({zero_shot_eval['n_samples']} samples)\n")
print(zero_shot_eval['classification_report'])


Zero-shot LLM accuracy: 41.2% (80 samples)

                   precision    recall  f1-score   support

    contamination       0.12      0.40      0.18         5
      deformation       0.38      0.60      0.46        15
             good       1.00      0.07      0.12        15
     other_defect       0.00      0.00      0.00        15
          scratch       0.62      0.87      0.72        15
structural_damage       0.47      0.53      0.50        15

         accuracy                           0.41        80
        macro avg       0.43      0.41      0.33        80
     weighted avg       0.47      0.41      0.35        80



## 4. Trained classifier (main approach)

A frozen, pretrained ResNet18 (`torchvision`, no fine-tuning) turns each training image into a 512-d feature vector; a `LogisticRegression` is trained on top of those vectors. Feature extraction is cached to disk (`extract_features_batch`) so re-running this cell later doesn't reprocess images already seen. Evaluated on the **full** test split — unlike the LLM baseline, this is free and local.

In [7]:
train_rows = [r for r in manifest_rows if r['split'] == 'train']
train_paths = [PROJECT_ROOT / r['filepath'] for r in train_rows]
train_labels = [r['coarse_label'] for r in train_rows]

print(f'Extracting features for {len(train_paths)} training images (cached after first run)...')
train_features = extract_features_batch(train_paths)
print(f'Feature matrix: {train_features.shape}')

clf = train_classifier(train_features, train_labels)
save_classifier(clf)
print('Classifier trained and saved to data/vision_model/classifier.joblib')


Extracting features for 1126 training images (cached after first run)...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/marcelocorreia/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:08<00:00, 5.46MB/s]


Feature matrix: (1126, 512)
Classifier trained and saved to data/vision_model/classifier.joblib


In [8]:
trained_eval = evaluate_classifier(
    lambda path: classify_defect_trained(path, clf), test_rows
)
print(f"Trained classifier accuracy: {trained_eval['accuracy']:.1%} "
      f"({trained_eval['n_samples']} samples, full test split)\n")
print(trained_eval['classification_report'])

from sklearn.metrics import confusion_matrix
y_true = [r['coarse_label'] for r in test_rows]
y_pred = [p['predicted_label'] for p in trained_eval['predictions']]
labels_sorted = sorted(set(y_true) | set(y_pred))
cm = confusion_matrix(y_true, y_pred, labels=labels_sorted)
print('Confusion matrix (rows=true, cols=predicted):')
print(f"{'':<20}" + ''.join(f'{l[:10]:>12}' for l in labels_sorted))
for label, row in zip(labels_sorted, cm):
    print(f'{label:<20}' + ''.join(f'{v:>12}' for v in row))


Trained classifier accuracy: 82.2% (376 samples, full test split)

                   precision    recall  f1-score   support

    contamination       0.29      0.40      0.33         5
      deformation       0.70      0.78      0.74        18
             good       0.90      0.89      0.90       290
     other_defect       0.61      0.55      0.58        20
          scratch       0.63      0.67      0.65        18
structural_damage       0.46      0.48      0.47        25

         accuracy                           0.82       376
        macro avg       0.60      0.63      0.61       376
     weighted avg       0.83      0.82      0.82       376

Confusion matrix (rows=true, cols=predicted):
                      contaminat  deformatio        good  other_defe     scratch  structural
contamination                  2           0           3           0           0           0
deformation                    1          14           2           0           1           0
good           

## 5. Benchmark: trained classifier vs. zero-shot LLM vs. majority-class

The required "benchmark against a baseline" comparison for this project's evaluation. A majority-class baseline ("always predict `good`") is included too — `good` images dominate the dataset (~77%), so this number shows how much of the trained classifier's accuracy is actually earned versus free from class imbalance.

In [9]:
majority_label = Counter(r['coarse_label'] for r in train_rows).most_common(1)[0][0]
majority_accuracy = sum(1 for r in test_rows if r['coarse_label'] == majority_label) / len(test_rows)

print(f'{"Approach":<28}{"Accuracy":>10}{"Samples":>10}   Notes')
print(f'{"Majority class (" + majority_label + ")":<28}{majority_accuracy:>10.1%}{len(test_rows):>10}   Trivial reference, no model at all')
print(f'{"Zero-shot LLM (baseline)":<28}{zero_shot_eval["accuracy"]:>10.1%}{zero_shot_eval["n_samples"]:>10}   gpt-4.1-mini, no training data needed, per-call API cost')
print(f'{"Trained classifier (main)":<28}{trained_eval["accuracy"]:>10.1%}{trained_eval["n_samples"]:>10}   Frozen ResNet18 features + LogisticRegression, free/local inference')


Approach                      Accuracy   Samples   Notes
Majority class (good)            77.1%       376   Trivial reference, no model at all
Zero-shot LLM (baseline)         41.2%        80   gpt-4.1-mini, no training data needed, per-call API cost
Trained classifier (main)        82.2%       376   Frozen ResNet18 features + LogisticRegression, free/local inference


## Milestone checkpoint

Done: `factory_floor/defect_dataset.py` (dataset scan + stratified manifest) and `factory_floor/vision.py` (`classify_defect_trained`, `classify_defect_zero_shot`, `classify_defect` dispatcher, `evaluate_classifier`) are the Vision Tool. Classification accuracy is reported on a held-out test split, benchmarked against a zero-shot LLM baseline and a majority-class reference — satisfying the project's multimodal-core and baseline-benchmarking requirements for this component.

Deliberately **not** built here (separate, later roadmap steps — see README's Compliance checklist and `dificuldades_e_oportunidades.md`):
- The Orchestrator Agent that would combine this Vision Tool with the RAG Tool and the History Tool into one diagnostic reasoning flow.
- Nameplate/fault-code photo reading (the original, incorrect interpretation of this roadmap item) — not part of the graded multimodal core, could be a future extra.
- Tracing/observability, the ≥30-scenario evaluation set with baseline benchmarking at the RAG level, and the Safety Data Sheet corpus — all separate open gaps, not this milestone.

In the Streamlit app, a new panel lets the operator upload a photo of a component and get its defect classification directly, independent of the RAG question form.